*** Modeling

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, roc_curve


pathdir = r"C:\Users\brisa\telco-customer-churn-analysis\data\processed"
X_train = pd.read_csv(f"{pathdir}\\X_train_final.csv")
y_train= pd.read_csv(f"{pathdir}\\y_train_final.csv").values.ravel() 
X_test =pd.read_csv(f"{pathdir}\\X_test_final.csv" ) ##data loadin
y_test = pd.read_csv(f"{pathdir}\\y_test_final.csv").values.ravel()

## logistic regression model as the baseline model
logModel= LogisticRegression(random_state=42, max_iter=1000)
logModel.fit(X_train,y_train) ##fit on training set
log_preds =logModel.predict(X_test ) ##predict on test set
logProbs = logModel.predict_proba(X_test )[:, 1] #propabilities will be needed ater for ROC curve
##eval
print("classification report for log reg:\n",classification_report(y_test, log_preds))
print(f"ROC-AUC Score for log reg = {roc_auc_score(y_test, logProbs):.4f}\n")

classification report for log reg:
               precision    recall  f1-score   support

           0       0.88      0.75      0.81      1035
           1       0.50      0.71      0.59       374

    accuracy                           0.74      1409
   macro avg       0.69      0.73      0.70      1409
weighted avg       0.78      0.74      0.75      1409

ROC-AUC Score for log reg = 0.8190



In [12]:
## Random Forest Classifier

##parameters that need to be evaluated to be set
paramTest= {
    'n_estimators' : [100, 200],
    'max_depth': [4, 6, 8], ## none is noo longer an option
    'min_samples_split' : [5, 10],
    'min_samples_leaf' : [2, 4]
}

## begin the grid search
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    paramTest ,
    cv= 5, ##five fold cross validation
    scoring='roc_auc',
    n_jobs= -1 
)

grid_search.fit(X_train, y_train) ##perform the grid search ont the training set
bestRf =grid_search.best_estimator_ ##grab results

print(f"Paramter tsting results: {grid_search.best_params_}")


rfPreds =bestRf.predict(X_test) ##eval model predictions on test set
rfprobs = bestRf.predict_proba(X_test)[:, 1] ##probabilities needed for ROC score

print("\nClassification Report for rf:\n", classification_report(y_test, rfPreds))
print(f"ROC-AUC Score for rf:{roc_auc_score(y_test, rfprobs):.4f}")

Paramter tsting results: {'max_depth': 8, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 200}

Classification Report for rf:
               precision    recall  f1-score   support

           0       0.91      0.75      0.82      1035
           1       0.53      0.79      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.77      0.73      1409
weighted avg       0.81      0.76      0.77      1409

ROC-AUC Score for rf:0.8414


Parameters need to be tuned.